# Combined ClinVar + UniProt RAG — query & summarize

Pipeline: **ingest both sources** → **index both** → **combined search** → **summarize**.

Prerequisites:
- `python src/ingest_clinvar.py` then `python src/build_clinvar_index.py`
- `python src/ingest_uniprot.py` then `python src/build_uniprot_index.py`

Single-source notebooks: `clinvar_rag.ipynb`, `uniprot_rag.ipynb`.

In [64]:
# Natural-language query
query = "What can we learn about telomere extension?"

# "BRCA1 hereditary breast cancer pathogenic" → top 3 ClinVar variants (lowest distances)
# "CFTR cystic fibrosis" → mixed ranking: 2 ClinVar + 1 UniProt
# "mismatch repair mechanism" → top 3 UniProt variants 

In [65]:
# Optional: install dependencies if needed
# !pip install chromadb sentence-transformers transformers huggingface_hub accelerate

In [66]:
# Imports
from pathlib import Path
import sys

from sentence_transformers import SentenceTransformer

In [67]:
# Project paths and shared RAG helpers
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root / "src"))

In [68]:
from rag_search import search_combined
from rag_summary import load_llm, summarize
from rag_export import append_rag_report

## Combined search

**Ranking:** Each source is queried independently (`TOP_K_CHUNKS` chunks per collection). Within each source, records are scored by their best (lowest-distance) chunk. Those per-record scores are merged and sorted globally; the top `TOP_K_RESULTS` records across ClinVar and UniProt are sent to the LLM (one full parquet row each).

In [69]:
# Retrieval limits
TOP_K_CHUNKS = 30  # chunk pool per collection; used for ranking (not sent to LLM)
TOP_K_RESULTS = 10 # 3  # top records globally across both sources

MIN_SIMILARITY = 0.65 # 0.55  
# Loose relevance gate as cosine similarity in [0,1] (higher = more relevant; matches the score shown to the LLM). 
# Calibrated on this corpus: real queries ~0.67-0.86, off-topic/nonsense ~0.41-0.53. 
# 0.55 passes every in-scope query while dropping clear nonsense -> empty context -> LLM abstains.

In [70]:
# Paths, embedding model, per-source search configs
data_processed = project_root / "data/processed"
chroma_dir = project_root / "data/chroma_db"

EMBED_MODEL = "BAAI/bge-small-en-v1.5"

CLINVAR_PARQUET = data_processed / "clinvar_rag.parquet"
CLINVAR_COLLECTION = "clinvar_chunks"
CLINVAR_RECORD_FIELDS = [
    "VariationID", "Name", "GeneSymbol", "Chromosome", "Start", "Stop",
    "Type", "ClinicalSignificance", "ReviewStatus", "PhenotypeList",
    "PhenotypeIDS", "Assembly", "HGNC_ID", "NumberSubmitters", "LastEvaluated",
]

UNIPROT_PARQUET = data_processed / "uniprot_rag.parquet"
UNIPROT_COLLECTION = "protein_chunks"
UNIPROT_RECORD_FIELDS = [
    "Entry", "Entry Name", "Gene Names", "Protein names", "Length",
    "Function [CC]", "Involvement in disease", "Subcellular location [CC]",
    "Domain [FT]", "Region", "Zinc finger", "Domain [CC]",
    "Natural variant", "Polymorphism", "Tissue specificity",
    "Developmental stage", "Interacts with",
]

SEARCH_CONFIGS = {
    "clinvar": {
        "collection_name": CLINVAR_COLLECTION,
        "parquet_path": CLINVAR_PARQUET,
        "group_key": "variation_id",
        "record_id_column": "VariationID",
        "record_id_meta_key": "variation_id",
        "record_fields": CLINVAR_RECORD_FIELDS,
        "section_header_template": "### Result {rank} — variation_id={variation_id}, gene={gene}",
        "hit_fields": [("variation_id", "variation_id"), ("gene", "gene")],
        "full_record_label": "Full ClinVar record",
        "record_id_cast": int,
    },
    "uniprot": {
        "collection_name": UNIPROT_COLLECTION,
        "parquet_path": UNIPROT_PARQUET,
        "group_key": "entry_name",
        "record_id_column": "Entry",
        "record_id_meta_key": "accession",
        "record_fields": UNIPROT_RECORD_FIELDS,
        "section_header_template": "### Result {rank} — entry_name={entry_name}, accession={accession}, gene={gene}",
        "hit_fields": [("entry_name", "entry_name"), ("accession", "accession"), ("gene", "gene")],
        "full_record_label": "Full UniProt record",
    },
}

In [71]:
embed_model = SentenceTransformer(EMBED_MODEL)
result = search_combined(
    query,
    SEARCH_CONFIGS,
    top_k_chunks=TOP_K_CHUNKS,
    top_k_results=TOP_K_RESULTS,
    chroma_dir=chroma_dir,
    embed_model=embed_model,
    min_similarity=MIN_SIMILARITY,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Collection clinvar_chunks: 11591 chunks
Collection protein_chunks: 19201 chunks

COMBINED RETRIEVAL
Query: What can we learn about telomere extension?
  clinvar: top 30 chunks → 30 chunk(s) across 30 record(s)
  uniprot: top 30 chunks → 30 chunk(s) across 18 record(s)

Combined ranking: each record's best (lowest-distance) chunk is scored, then the top 10 records across all sources are sent to the LLM (one full parquet row per record).
Top 10 breakdown: uniprot=10
  1. [uniprot] TELT_HUMAN gene=TCAP similarity=0.686
  2. [uniprot] TELO2_HUMAN gene=TELO2 similarity=0.682
  3. [uniprot] TERB2_HUMAN gene=TERB2 similarity=0.682
  4. [uniprot] ACD_HUMAN gene=ACD similarity=0.677
  5. [uniprot] TINF2_HUMAN gene=TINF2 similarity=0.670
  6. [uniprot] RTEL1_HUMAN gene=RTEL1 similarity=0.670
  7. [uniprot] POTE1_HUMAN gene=POT1 similarity=0.669
  8. [uniprot] TTI2_HUMAN gene=TTI2 similarity=0.665
  9. [uniprot] TCAB1_HUMAN gene=WRAP53 similarity=0.663
  10. [uniprot] NSE2_HUMAN gene=NSMCE2 simil

## Summarize

In [72]:
SYSTEM_PROMPT = (
    "You are a clinical genomics and protein biology assistant. "
    "Write a brief summary in plain English sentences that answers the user's query using ONLY the ClinVar and UniProt records provided. "
    "Clearly distinguish variant evidence (ClinVar: cite VariationID and Name) from protein evidence (UniProt: cite Entry Name and Protein Name). "
    "Do not invent genes, variants, proteins, significance, or other facts not present in the provided text. "
    "Each record is tagged with a relevance score (cosine_similarity in [0,1]; higher = closer to the query). "
    "Treat low-similarity records with caution and rely on them only if they genuinely address the query. "
    "If the evidence states that no relevant records were retrieved, or none of the records actually address the query, "
    "say plainly that the indexed data does not contain an answer rather than guessing. "
    "If information is missing, say so briefly."
)

Load summarizer → generate answer from retrieved context only.

- **local:** downloads once → cached under `~/.cache/huggingface/`; 0.5B ~2GB RAM.
- **openai:** calls `gpt-4o-mini` via API; set `OPENAI_API_KEY` in `.env`.

The LLM receives the top 3 records from the combined ranking (ClinVar and/or UniProt), not the raw chunk pool.

In [73]:
# Summarization backend: "local" or "openai"
SUMMARIZER_BACKEND = "openai"
# Local model (SUMMARIZER_BACKEND == "local")
# Needs ~8GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
# Needs ~4GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# OpenAI model (SUMMARIZER_BACKEND == "openai")
API_MODEL = "gpt-4o-mini"
MAX_NEW_TOKENS = 1024
REPORT_PATH = project_root / "reports/rag_log.md"
CSV_LOG_PATH = project_root / "data/reports/rag_log.csv"

In [74]:
if SUMMARIZER_BACKEND == "local":
    llm_model, llm_tokenizer = load_llm(
        LLM_MODEL,
        model=globals().get("llm_model"),
        tokenizer=globals().get("llm_tokenizer"),
    )
else:
    llm_model, llm_tokenizer = None, None

In [75]:
if not result["ranked"]:
    # Relevance gate dropped everything → abstain without spending an LLM call.
    summary = (
        "The indexed ClinVar + UniProt data does not contain an answer to this query."
    )
else:
    summary = summarize(
        query,
        result["context"],
        llm_model,
        llm_tokenizer,
        SYSTEM_PROMPT,
        max_new_tokens=MAX_NEW_TOKENS,
        backend=SUMMARIZER_BACKEND,
        api_model=API_MODEL,
    )
print("=" * 60)
print(f"Query: {query}\n")
print(summary)

_ = append_rag_report(
    REPORT_PATH,
    query,
    result,
    summary,
    search_mode="combined",
    summarizer_backend=SUMMARIZER_BACKEND,
    model_name=API_MODEL if SUMMARIZER_BACKEND == "openai" else LLM_MODEL,
    csv_path=CSV_LOG_PATH,
)

Summarized via OpenAI: gpt-4o-mini
Query: What can we learn about telomere extension?

From the retrieved evidence, we can learn that telomere extension is influenced by several proteins involved in telomere maintenance and regulation. 

1. **UniProt Entry: TELO2_HUMAN (Q9Y4R8)** - This protein, known as the Telomere length regulation protein TEL2 homolog, is involved in stabilizing proteins that are crucial for the DNA damage response and may play a role in telomere length regulation.

2. **UniProt Entry: ACD_HUMAN (Q96AP0)** - The Adrenocortical dysplasia protein homolog is part of the shelterin complex, which protects telomeres and regulates their length. It enhances telomere elongation by recruiting telomerase to telomeres.

3. **UniProt Entry: TINF2_HUMAN (Q9BSI4)** - This protein is also a component of the shelterin complex and plays a role in telomere length regulation and protection.

4. **UniProt Entry: RTEL1_HUMAN (Q9NZ71)** - The Regulator of telomere elongation helicase 1 i